In [2]:
"""
Imputation pipeline for sparse, irregularly-sampled ICU ventilator time series.

Design principles (see chat discussion):
  1. Missingness is informative -> keep explicit mask + time-since-last-observed
     features instead of silently erasing the fact that something was missing.
  2. Forward-fill is only clinically valid WITHIN a patient stay, and only up to
     a bounded time window (a setting from 6 hours ago is not "current").
  3. Columns that are structurally missing for an entire stay (device not used)
     should not be statistically imputed from other columns in that same stay -
     they get flagged, then optionally backed off to a reference population value.
  4. Any reference statistics (medians, etc.) must be computed on a TRAIN split
     only in a real project, then re-used (not recomputed) on val/test to avoid
     leakage. This script exposes that as an explicit `reference_stats` argument.
"""

import pandas as pd
import numpy as np

ID_COL = "stay_id"
TIME_COL = "charttime"
# Max time (in minutes) a value is allowed to be carried forward before it's
# considered stale again. Tune per-variable in a real project -- e.g. FiO2
# might be trusted for longer than a fast-moving value like SpO2.
DEFAULT_LOCF_CUTOFF_MIN = 240  # 4 hours


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)[:40000]
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
    return df


def add_mask_and_delta_features(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """Adds <col>_observed (1/0) and <col>_delta_t_min (minutes since the last
    real observation of that column, within the same stay) BEFORE any filling
    is done, so these reflect the true measurement pattern."""
    df = df.copy()
    for col in value_cols:
        df[f"{col}_observed"] = df[col].notna().astype(int)

        delta = np.full(len(df), np.nan)
        for stay, idx in df.groupby(ID_COL).groups.items():
            idx = list(idx)
            last_seen_time = None
            for i in idx:
                t = df.at[i, TIME_COL]
                if last_seen_time is not None:
                    delta[i] = (t - last_seen_time).total_seconds() / 60.0
                if df.at[i, col] == df.at[i, col]:  # not NaN
                    last_seen_time = t
        df[f"{col}_delta_t_min"] = delta
    return df


def time_aware_locf(df: pd.DataFrame, value_cols: list,
                     cutoff_min: float = DEFAULT_LOCF_CUTOFF_MIN) -> pd.DataFrame:
    """Forward-fills each value column within its own stay_id, but only while
    the gap since the last real reading stays under `cutoff_min`. Once the gap
    exceeds the cutoff the value reverts to NaN (stale settings shouldn't be
    treated as current)."""
    df = df.copy()
    for col in value_cols:
        filled_col = f"{col}_locf"
        df[filled_col] = df[col]
        for stay, group in df.groupby(ID_COL):
            idx = group.index
            last_val, last_time = None, None
            for i in idx:
                t = df.at[i, TIME_COL]
                if pd.notna(df.at[i, col]):
                    last_val, last_time = df.at[i, col], t
                elif last_val is not None:
                    gap = (t - last_time).total_seconds() / 60.0
                    if gap <= cutoff_min:
                        df.at[i, filled_col] = last_val
                    # else leave as NaN -- too stale to carry forward
    return df


def flag_structurally_missing(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """Marks columns that are 100% missing for an ENTIRE stay -- these are
    candidates for 'device/measurement not used', not for statistical fill."""
    df = df.copy()
    for col in value_cols:
        df[f"{col}_structurally_missing_in_stay"] = (
            df.groupby(ID_COL)[col].transform(lambda s: s.notna().sum() == 0).astype(int)
        )
    return df


def reference_fallback_fill(df: pd.DataFrame, value_cols: list,
                             reference_stats: dict) -> pd.DataFrame:
    """Last resort fill for whatever is STILL missing after time-aware LOCF
    (e.g. before the first real reading in a stay, or fully-missing columns).
    `reference_stats` must come from a TRAIN split in real use -- passed in
    explicitly here rather than computed from this dataframe, to make that
    requirement unavoidable to see."""
    df = df.copy()
    for col in value_cols:
        locf_col = f"{col}_locf"
        final_col = f"{col}_final"
        fallback_val = reference_stats.get(col, np.nan)
        df[final_col] = df[locf_col].fillna(fallback_val)
    return df


def run_pipeline(path: str, reference_stats: dict,
                  cutoff_min: float = DEFAULT_LOCF_CUTOFF_MIN) -> pd.DataFrame:
    df = load_data(path)
    value_cols = [c for c in df.columns if c not in (ID_COL, TIME_COL)]

    df = add_mask_and_delta_features(df, value_cols)
    df = time_aware_locf(df, value_cols, cutoff_min)
    df = flag_structurally_missing(df, value_cols)
    df = reference_fallback_fill(df, value_cols, reference_stats)
    return df


if __name__ == "__main__":
    RAW_VALUE_COLS = [
        "spo2", "fio2", "flow_rate", "peep", "pip", "respiratory_rate_total",
        "minute_volume", "tidal_volume_observed", "etco2",
        "inspiratory_ratio", "expiratory_ratio",
    ]

    # In a real project: compute this dict from df_train[col].median() only.
    # Hardcoded here as illustrative clinical-population reference values.
    reference_stats = {
        "spo2": 97.0, "fio2": 40.0, "flow_rate": 8.0, "peep": 5.0, "pip": 20.0,
        "respiratory_rate_total": 16.0, "minute_volume": 7.0,
        "tidal_volume_observed": 450.0, "etco2": 38.0,
        "inspiratory_ratio": 1.0, "expiratory_ratio": 2.0,
    }

    result = run_pipeline("bq-res.csv", reference_stats)

    # Reorder columns for readability: id/time, then per-variable groups
    ordered = [ID_COL, TIME_COL]
    for col in RAW_VALUE_COLS:
        ordered += [col, f"{col}_observed", f"{col}_delta_t_min",
                    f"{col}_locf", f"{col}_structurally_missing_in_stay",
                    f"{col}_final"]
    result = result[ordered]

    result.to_csv("df_imputed.csv", index=False)
    print(result[[ID_COL, TIME_COL, "spo2", "spo2_observed",
                   "spo2_delta_t_min", "spo2_locf", "spo2_final",
                   "flow_rate", "flow_rate_structurally_missing_in_stay",
                   "flow_rate_final"]].to_string())
                   

        stay_id           charttime      spo2  spo2_observed  spo2_delta_t_min  spo2_locf  spo2_final  flow_rate  flow_rate_structurally_missing_in_stay  flow_rate_final
0      30000153 2174-09-29 12:00:00       NaN              0               NaN        NaN        97.0        NaN                                       1              8.0
1      30000153 2174-09-29 12:01:00       NaN              0               NaN        NaN        97.0        NaN                                       1              8.0
2      30000153 2174-09-29 12:05:00     100.0              1               NaN      100.0       100.0        NaN                                       1              8.0
3      30000153 2174-09-29 12:25:00       NaN              0              20.0      100.0       100.0        NaN                                       1              8.0
4      30000153 2174-09-29 13:00:00     100.0              1              55.0      100.0       100.0        NaN                                      